Absolute Positional Embedding

In [7]:
import torch
import tiktoken
from torch.utils.data import DataLoader, Dataset

In [ ]:
#total number of unique tokens that LLM can represent
vocab_size = 50257

#how many dimensions each token can be represented by (vector)
output_dim = 256

#Create embedding layer
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [9]:
#create a dataset class that takes in the text, tokenizer, max_length, and stride
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids =[]
        token_ids = tokenizer.encode(txt, allowed_special = {"<|endoftext|>"}) 

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
#Batch size = num of input sequence
#Context size = tokens per input sequence = max_length

def create_dataLoader_V1(txt, batch_size = 4, max_length = 256, stride = 128, shuffle = True, drop_last = True, num_workers = 0):

    #tokenizes using BPE
    tokenizer = tiktoken.get_encoding("gpt2")

    #creates a dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    #creates a DataLoader
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return data_loader

In [11]:
#read the text file
with open("the-verdict.txt", "r", encoding = "utf-8") as f:
    raw_text = f.read()

In [ ]:
#context size
max_length = 4

dataloader = create_dataLoader_V1(raw_text,batch_size = 8, max_length = max_length, stride = max_length, shuffle = False)

data_iter = iter(dataloader)

inputs,targets = next(data_iter)

In [18]:
print(inputs)

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])


In [19]:
token_embedding = token_embedding_layer(inputs)

print(token_embedding.shape)

print(token_embedding)

torch.Size([8, 4, 256])
tensor([[[ 0.2216, -0.2566,  0.0517,  ..., -0.1652, -0.9786,  0.9408],
         [ 0.3632, -0.9308,  2.0474,  ...,  1.4945, -0.1124,  1.0614],
         [-0.7056, -0.4923, -1.2773,  ...,  0.7643,  1.1340, -0.3084],
         [ 0.2793,  1.3721,  0.7079,  ..., -1.2065, -0.1918,  0.3629]],

        [[ 1.5040,  0.7135, -2.3280,  ...,  0.3851,  0.9475,  0.1349],
         [ 1.6185, -0.8104, -0.2958,  ..., -1.4405,  0.3974, -0.9348],
         [-0.2390, -0.7620,  0.8486,  ...,  1.2589,  2.9589,  0.4040],
         [ 0.7403,  0.1555,  1.6678,  ...,  0.0331, -1.2116,  1.2044]],

        [[ 0.7022,  0.5808,  0.0257,  ...,  0.0306,  0.7925,  0.0114],
         [-0.4783, -0.2725,  1.7023,  ..., -0.5497,  0.1864, -0.3609],
         [-1.4754,  0.8612,  0.5363,  ..., -0.4608,  0.4725,  0.1974],
         [ 0.2747,  0.1283, -0.6424,  ...,  1.1979,  0.3184,  0.7491]],

        ...,

        [[ 1.0524, -0.0877,  0.9537,  ...,  1.5736, -0.0554, -1.0158],
         [-0.3580,  0.7450, -1.26

To put into perspective, our input size is 8 (rows) x 4 (tokens at a time). Each element in this matrix can be represented as a vector thats about 256 dimension, explaining the shape.

Now, to create an embedding layer for positional embedding

In [20]:
context_len = max_length

positional_embed_layer = torch.nn.Embedding(context_len, output_dim)

positional_embeds = positional_embed_layer(torch.arange(max_length))

print(positional_embeds.shape)

torch.Size([4, 256])
